In [0]:
# Databricks notebook source
import os
import shutil

# Obtener parámetros de widgets
dbutils.widgets.text("catalog", "datathon")
dbutils.widgets.text("schema", "gcc")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

# 1. Crear catálogo, esquema y volumen en Unity Catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.raw")

# 2. Rutas de origen y destino en el filesystem
# Al ejecutarse en Databricks, la ruta de trabajo actual es la carpeta src/
current_dir = os.getcwd()
source_path = os.path.normpath(os.path.join(current_dir, "..", "data", "datathon_data_gcc.csv"))
target_volume_dir = f"/Volumes/{catalog}/{schema}/raw"
target_path = os.path.join(target_volume_dir, "datathon_data_gcc.csv")

print(f"Buscando archivo de datos en: {source_path}")

# Si por alguna razón la ruta relativa cambia según el entorno de ejecución
if not os.path.exists(source_path):
    # Buscar alternativamente en la misma carpeta o subdirectorio
    alt_path = os.path.join(current_dir, "data", "datathon_data_gcc.csv")
    if os.path.exists(alt_path):
        source_path = alt_path

if not os.path.exists(source_path):
    raise FileNotFoundError(f"No se encontro el archivo en {source_path}. Verifica que este en la carpeta data/ del repositorio.")

# 3. Copiar directamente al volumen
print(f"Copiando a Unity Catalog Volume: {target_path}")
shutil.copyfile(source_path, target_path)
print("Archivo preparado exitosamente en Unity Catalog.")